# Develop, test, and deploy a quantum error correction scheme with `qdk.ec`

Taking a quantum error correction scheme from a paper to a production pipeline is
hard. It usually means writing a bespoke simulation to convince yourself the scheme
works, and then coordinating with several teams to teach a compilation pipeline
about it.

`qdk.ec` closes that gap around one artifact: a **qodec**. A qodec is a declarative
description of a compilation pipeline together with the error correction schemes
that lower each layer of it. Because it is *just data*, the same file you test
against a local simulator is the file you hand to the compilation pipeline.

This notebook walks the three stages the package is organised around:

| stage | subpackage | question it answers |
| --- | --- | --- |
| develop | `qdk.ec.develop` | how do I load, save, and finish a qodec? |
| test | `qdk.ec.profile`, `qdk.ec.audit` | what does this qodec actually do, and is that what I meant? |
| deploy | `qdk.ec.targets` | what happens when I run it on a real backend? |

## Installing

`qdk.ec` is an optional extra of the `qdk` package:

```bash
pip install "qdk[ec]"            # authoring + analysis
pip install "qdk[ec,ec-backends]"  # ... plus the stim / mwpf backends used below
```


## 1. Develop — load a qodec

`qdk.ec.develop` holds the primitives that move qodecs between disk, memory, and
YAML text. We start from `c4.qodec.yaml`, sitting next to this notebook: the
[[4,2,2]] error-*detecting* code, which encodes two logical qubits in four
physical ones and can detect (but not correct) any single-qubit fault.

In [1]:
from qdk.ec import audit, develop, profile, targets

codec = develop.load("c4.qodec.yaml")
print(codec.summary())

ImportError: dynamic module does not define module export function (PyInit_qodec)

A qodec is a chain of **layers**, from the most abstract instruction set down to
the most concrete. Each layer carries the **gadgets** that lower one of its
instructions into a circuit over the layer below. Here there is a single lowering
edge: the logical `C4` instruction set down to physical `stim` operations.

In [ ]:
layer = codec.layers[0]
print("lowering:", layer.isa.name, "->", codec.layers[1].isa.name)
print("gadgets: ", sorted(layer.gadgets))

## 2. Profile — characterise the code

`qdk.ec.profile` computes focused, typed characteristics of qodec objects. Start
with the code itself: its stabilizers, its logical operators, and its distance.

In [ ]:
code = codec.codes["C4"]

print("stabilizers:", list(code.stabilizers))
print("logical X:  ", list(code.x))
print("logical Z:  ", list(code.z))

distance, witness = profile.code_distance_of(code)
print(f"distance:    {distance}  (witness: {[str(p) for p in witness]})")

Distance 2 is exactly what "error *detecting*" means: there is a weight-2 logical
error, so a single fault is always visible but never correctable.

### Declared vs. realized action

Every gadget makes a promise — the action of the instruction it `implements` — and
keeps it with a circuit. Those are two independent objects, and `qdk.ec` can
compute both and compare them. This is the check that catches a transcription slip
between the paper and the circuit.

In [ ]:
measure_zz = layer.gadgets["measure_zz"]

print("declared:", profile.declared_action_of(measure_zz))
print("realized:", profile.realized_action_of(measure_zz))
print("mismatch:", profile.gadget_action_mismatch(measure_zz) or "none")

### Checks and readouts

A gadget's circuit produces raw measurement outcomes. Two derived structures give
those outcomes meaning:

* **checks** — parities of outcomes that are *deterministic*, so a flip signals a
  fault. These are what a decoder consumes.
* **readouts** — the parities that carry the logical answer the instruction
  promised.

Both are discovered by exact simulation, so you never have to derive them by
hand.

In [ ]:
discovered = profile.readouts.profile_of(measure_zz)

print("checks:     ", discovered.checks)
print("observables:", discovered.observables)
print("essential:  ", profile.essential_checks_of(measure_zz))

## 3. Develop — let the tooling finish the draft

Because checks and readouts are *derivable*, an author should not have to write
them. `develop.complete_gadget` fills them in for one gadget, and
`develop.complete_qodec` does it for an entire qodec.

To show it working, take a gadget, throw its checks away, and ask `qdk.ec` to put
them back.

In [ ]:
import qodec

draft = qodec.Gadget(
    measure_zz.implements,
    measure_zz.circuit,
    inputs=list(measure_zz.inputs),
    outputs=list(measure_zz.outputs),
    checks=[],
    readouts=[[str(atom) for atom in entry] for entry in measure_zz.readouts],
)
print("draft checks:    ", list(draft.checks))

completed = develop.complete_gadget(draft)
print("completed checks:", [[str(atom) for atom in check] for check in completed.checks])

`complete_qodec` applies the same treatment to every gadget of every layer, and
returns a new qodec — the input is never mutated.

In [ ]:
completed_codec = develop.complete_qodec(codec)

for mnemonic, gadget in sorted(completed_codec.layers[0].gadgets.items()):
    print(f"{mnemonic:16s} {len(gadget.checks)} check(s)")

### Round-tripping through YAML

A qodec is data, so it round-trips. `to_yaml` / `from_yaml` keep it in memory;
`save` / `load` put it on disk. This is the handoff to the compilation pipeline:
the artifact you just tested *is* the deployment config.

In [ ]:
text = develop.to_yaml(completed_codec)
print(f"{len(text)} characters of YAML, {len(text.splitlines())} lines")

reloaded = develop.from_yaml(text)
print("round-trips:", reloaded.name == completed_codec.name)

## 4. Test — audit the qodec

`qdk.ec.audit` runs a rule set over the whole qodec and returns structured
diagnostics: each one names the rule that fired, the object it fired on, and why.
This is the "did I write what I meant?" pass.

In [ ]:
report = audit.audit(codec)
print(f"{len(report.errors())} error(s), {len(report.warnings())} warning(s)")

for diagnostic in report.errors() + report.warnings()[:2]:
    print()
    print(f"[{diagnostic.severity.name}] {diagnostic.rule}")
    print(f"  {diagnostic.summary}")

The report flags two kinds of problem here, and both are the kind that is
invisible in a paper and fatal in a pipeline: `measure_xx` declares readout
parities that its own circuit does not produce, and several gadgets never declare
a sign for their output stabilizers, so a decoder cannot tell which frame it is
being handed.

### Equivalence

The other half of testing is comparison: is this refactored gadget the same as the
one I trust? `qdk.ec.audit.equivalence` answers that, and explains a "no".

In [ ]:
measure_xx = layer.gadgets["measure_xx"]

print("measure_zz == itself:    ", audit.gadgets_equivalent(measure_zz, measure_zz))
print("measure_zz == measure_xx:", audit.gadgets_equivalent(measure_zz, measure_xx))
print("why not:", audit.why_not_equivalent(measure_zz, measure_xx))

## 5. Deploy — run it on a target

A **target** takes a qodec plus a program written in its most abstract instruction
set, and does something with them: sample it, build a detector error model,
estimate resources. `qdk.ec.targets` ships a few, and `TargetModel` is the
protocol for building your own.

First, a program. It is written entirely in *logical* `C4` instructions — the
qodec knows how to lower it.

In [ ]:
from qodec.circuits import Program


def call(mnemonic: str) -> qodec.instructions.InstructionCall:
    """An InstructionCall binding every operand of `mnemonic` to one block."""
    instruction = layer.isa.instruction(mnemonic)
    inputs = {str(i): "q" for i in range(len(list(instruction.inputs)))}
    outputs = {str(i): "q" for i in range(len(list(instruction.outputs)))}
    if not inputs and not outputs:
        return qodec.instructions.InstructionCall(mnemonic)
    return qodec.instructions.InstructionCall(mnemonic, inputs=inputs, outputs=outputs)


program = Program([call(m) for m in ("prepare_zz", "idle", "measure_zz")], layer.isa)
print([c.mnemonic for c in program.instructions])

### Sampling

`StimSampler` lowers the logical program to a physical stim circuit and samples it.
Noiseless, the detectors must never fire — anything else is a bug in the qodec.

In [ ]:
import numpy as np

noiseless = targets.StimSampler(codec)
shots = np.asarray(noiseless.execute(program, shots=200))

events = noiseless.emitter.detection_events(program, shots)
print(f"{shots.shape[0]} shots x {shots.shape[1]} measurement records")
print("detection events fired:", int(events.sum()))

Turn the noise on and the same detectors start firing — the code is doing its
job.

In [ ]:
noisy = targets.StimSampler(codec, noise={"p_data": 0.01, "p_meas": 0.01})
noisy_shots = np.asarray(noisy.execute(program, shots=2000))

flagged = noisy.emitter.detection_events(program, noisy_shots).any(axis=1)
print(f"shots with at least one detection: {flagged.mean():.1%}")

### Detector error models

For decoding, what you want is not shots but a **detector error model**: the graph
of independent error mechanisms and the detectors each one flips.

In [ ]:
dem = targets.detector_error_model_of(
    codec, program, {"p_data": 0.001, "p_meas": 0.001}
)
print("\n".join(str(dem).splitlines()[:8]))

### Circuit-level distance

Code distance describes the code. What matters operationally is the distance of the
*gadget* under a concrete noise model — the smallest number of circuit faults that
produces an undetected logical error. For `measure_xx` it comes out at 2, matching
the code: the circuit does not squander the protection the code provides.

In [ ]:
model = targets.depolarizing(0.001)
gadget_distance, fault_witness = targets.gadget_distance_of(measure_xx, model)

print("gadget distance:", gadget_distance)
for fault in fault_witness:
    print(" ", fault)

## Where to go next

* `qdk.ec.develop` — `load`, `save`, `from_yaml`, `to_yaml`, `complete_gadget`,
  `complete_qodec`.
* `qdk.ec.profile` — `action`, `checks`, `code`, `distance`, `faults`, `readouts`.
* `qdk.ec.audit` — `audit`, `why_not_valid`, and the `equivalence` predicates.
* `qdk.ec.targets` — `TargetModel`, `StimSampler`, `PaulimerSampler`,
  `detector_error_model_of`, `gadget_distance_of`.

The qodec you finish here is the artifact you deploy: no rewrite, no second
implementation, no cross-team translation.